---
last_verified: 2026-09-02
tool_version: n/a
---

# Comparing ruff format vs black on the same codebase

> L3 investigation: what each formatter does to the same file, where their outputs disagree, and how to resolve the conflict so only one of them owns formatting.

## Purpose

Both `ruff format` and `black` reformat Python code, and running both in the same repo produces competing diffs. This notebook runs each formatter on one shared sample file, compares the outputs, and settles on a single-owner setup. The docs also suggest picking one formatter and letting the other tool stick to linting — that is the approach followed here.

## Step 1 — sample file both formatters will see

A short module with deliberately awkward spacing, mixed quote style, and an over-long line, so the formatting decisions are visible.

In [ ]:
# File: sample_format_me.py (before formatting)
def greet( name, greeting='hello' ):
    msg =  greeting + ", " +  name + "!"
    print( msg )
    return msg


def total(items):
    result = 0
    for item in items:
        result = result + item
    return result


x = greet( "world" )
print( total( [1, 2, 3] ) )

## Step 2 — format with ruff, then reset and format with black

Run one formatter at a time on a pristine copy so the second run never sees the first formatter's output. `--check` previews without writing; `--diff` shows the proposed edits.

In [ ]:
# Run:  cp sample_format_me.py /tmp/fmt_ruff.py
# Run:  ruff format --diff /tmp/fmt_ruff.py   # preview ruff's edits
# Run:  ruff format /tmp/fmt_ruff.py          # apply ruff's edits
#
# Run:  cp sample_format_me.py /tmp/fmt_black.py
# Run:  black --diff /tmp/fmt_black.py       # preview black's edits
# Run:  black /tmp/fmt_black.py              # apply black's edits
#
# Run:  diff /tmp/fmt_ruff.py /tmp/fmt_black.py   # the disagreement, if any

## Step 3 — where the outputs disagree

On this kind of sample the two outputs are close but not identical. Typical deltas:

1. Quote normalization — both prefer double quotes by default, but edge cases (quotes inside strings) can differ.
2. Line splitting — long call chains and long string concatenations get split at different points.
3. Scope — `ruff format` only reformats; import sorting lives in `ruff check` with the import rules selected. Black never sorts imports. So an unsorted-import diff from ruff is a lint fix (`ruff check --fix`), not a formatter disagreement.

The practical takeaway: the diffs are stylistic, not correctness issues. The conflict only hurts when both formatters run in CI and each re-dirties the other's output.

In [ ]:
# Run:  ruff check --select I /tmp/fmt_ruff.py   # import-order lint, separate from formatting
# Run:  ruff check --fix --select I /tmp/fmt_ruff.py
#
# If the only remaining diff between the two formatted files is
# whitespace/line-splitting, that confirms this is a formatter-ownership
# question, not a lint problem.

## Step 4 — resolve the conflict: one formatter owns formatting

Pick one and disable the other. Two workable setups:

- Option A (ruff only): run `ruff check` for lint plus `ruff format` for formatting; do not run black at all.
- Option B (black only): run black for formatting and restrict ruff to lint (`ruff check`), excluding its format step from hooks and CI.

This repo uses option A — one tool, fewer moving parts. The pre-commit/CI gate then runs `ruff check` followed by `ruff format --check`, and black is left out entirely.

In [ ]:
# Gate used after this decision (ruff owns formatting):
# Run:  ruff check sample_format_me.py
# Run:  ruff format --check sample_format_me.py
#
# Both commands exit non-zero on violations, so a CI step or
# pre-commit hook can chain them with && and fail the run.

## Verify

1. `ruff format --check` passes on the sample after `ruff format` is applied — the file is stable under the chosen formatter.
2. Re-running `ruff format` on its own output produces no further diff (idempotent).
3. `ruff check` reports no import-order or unused-import findings that could be mistaken for format drift.
4. Black is not part of the gate, so there is no second formatter to re-dirty the file — the original conflict is gone by construction.